### About Dataset

Context

This is a small subset of dataset of Book reviews from Amazon Kindle Store category.

Content 5-core dataset of product reviews from Amazon Kindle Store category from May 1996 - July 2014. Contains total of 982619 entries. Each reviewer has at least 5 reviews and each product has at least 5 reviews in this dataset. Columns

asin - ID of the product, like B000FA64PK
helpful - helpfulness rating of the review - example: 2/3.
overall - rating of the product.
reviewText - text of the review (heading).
reviewTime - time of the review (raw).
reviewerID - ID of the reviewer, like A3SPTOKDG7WBLN
reviewerName - name of the reviewer.
summary - summary of the review (description).
unixReviewTime - unix timestamp.
Acknowledgements This dataset is taken from Amazon product data, Julian McAuley, UCSD website. http://jmcauley.ucsd.edu/data/amazon/

License to the data files belong to them.

Inspiration

Sentiment analysis on reviews.
Understanding how people rate usefulness of a review/ What factors influence helpfulness of a review.
Fake reviews/ outliers.
Best rated product IDs, or similarity between products based on reviews alone (not the best idea ikr).
Any other interesting analysis
### Best Practises
1.Preprocessing And Cleaning

2.Train Test Split

3.BOW,TFIDF,Word2vec

4.Train ML algorithms


In [19]:
import pandas as pd
data=pd.read_csv("/content/sample_data/all_kindle_review.csv")
data.head()

,Unnamed: 0.1,Unnamed: 0,asin,helpful,rating,reviewText,reviewTime,reviewerID,reviewerName,summary,unixReviewTime
0,0,11539,B0033UV8HI,"[8, 10]",3,"Jace Rankin may be short, but he's nothing to ...","09 2, 2010",A3HHXRELK8BHQG,Ridley,Entertaining But Average,1283385600
1,1,5957,B002HJV4DE,"[1, 1]",5,Great short read. I didn't want to put it dow...,"10 8, 2013",A2RGNZ0TRF578I,Holly Butler,Terrific menage scenes!,1381190400
2,2,9146,B002ZG96I4,"[0, 0]",3,I'll start by saying this is the first of four...,"04 11, 2014",A3S0H2HV6U1I7F,Merissa,Snapdragon Alley,1397174400
3,3,7038,B002QHWOEU,"[1, 3]",3,Aggie is Angela Lansbury who carries pocketboo...,"07 5, 2014",AC4OQW3GZ919J,Cleargrace,very light murder cozy,1404518400
4,4,1776,B001A06VJ8,"[0, 1]",4,I did not expect this type of book to be in li...,"12 31, 2012",A3C9V987IQHOQD,Rjostler,Book,1356912000


In [20]:
data=data[['reviewText','rating']]
data.head()

,reviewText,rating
0,"Jace Rankin may be short, but he's nothing to ...",3
1,Great short read. I didn't want to put it dow...,5
2,I'll start by saying this is the first of four...,3
3,Aggie is Angela Lansbury who carries pocketboo...,3
4,I did not expect this type of book to be in li...,4


In [21]:
data.shape

(12000, 2)

In [22]:
data.isnull().sum(
)

,0
reviewText,0
rating,0


In [23]:
data['rating'].value_counts()

,count
rating,
5,3000
4,3000
3,2000
2,2000
1,2000


In [24]:
#Preprocessing and Cleaning

In [25]:
data['rating']=data['rating'].apply(lambda x: 0 if x<3 else 1)

In [26]:
df=data

In [27]:
df.head()

,reviewText,rating
0,"Jace Rankin may be short, but he's nothing to ...",1
1,Great short read. I didn't want to put it dow...,1
2,I'll start by saying this is the first of four...,1
3,Aggie is Angela Lansbury who carries pocketboo...,1
4,I did not expect this type of book to be in li...,1


In [29]:
df['rating'].value_counts()

,count
rating,
1,8000
0,4000


In [30]:
df['reviewText']=df['reviewText'].str.lower()

In [31]:
df.head()

,reviewText,rating
0,"jace rankin may be short, but he's nothing to ...",1
1,great short read. i didn't want to put it dow...,1
2,i'll start by saying this is the first of four...,1
3,aggie is angela lansbury who carries pocketboo...,1
4,i did not expect this type of book to be in li...,1


In [33]:
## Removing special characters
import re
import nltk
from nltk.corpus import stopwords
from bs4 import BeautifulSoup
nltk.download('stopwords')
df['reviewText']=df['reviewText'].apply(lambda x:re.sub('[^a-z A-z 0-9-]+', '',x))
## Remove the stopswords
df['reviewText']=df['reviewText'].apply(lambda x:" ".join([y for y in x.split() if y not in stopwords.words('english')]))
## Remove url
df['reviewText']=df['reviewText'].apply(lambda x: re.sub(r'(http|https|ftp|ssh)://([\w_-]+(?:(?:\.[\w_-]+)+))([\w.,@?^=%&:/~+#-]*[\w@?^=%&/~+#-])?', '' , str(x)))
## Remove html tags
df['reviewText']=df['reviewText'].apply(lambda x: BeautifulSoup(x, 'lxml').get_text())
## Remove any additional spaces
df['reviewText']=df['reviewText'].apply(lambda x: " ".join(x.split()))

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


In [34]:
df.head()

,reviewText,rating
0,jace rankin may short hes nothing mess man hau...,1
1,great short read didnt want put read one sitti...,1
2,ill start saying first four books wasnt expect...,1
3,aggie angela lansbury carries pocketbooks inst...,1
4,expect type book library pleased find price right,1


In [35]:
#Lemmatizer
from nltk.stem import WordNetLemmatizer
nltk.download('wordnet')
lemmatizer=WordNetLemmatizer()


[nltk_data] Downloading package wordnet to /root/nltk_data...


In [36]:

def lemmatize_words(text):
    return " ".join([lemmatizer.lemmatize(word) for word in text.split()])

In [37]:
df['reviewText']=df['reviewText'].apply(lambda x:lemmatize_words(x))

In [38]:
df.head()

,reviewText,rating
0,jace rankin may short he nothing mess man haul...,1
1,great short read didnt want put read one sitti...,1
2,ill start saying first four book wasnt expecti...,1
3,aggie angela lansbury carry pocketbook instead...,1
4,expect type book library pleased find price right,1


In [39]:
from sklearn.model_selection import train_test_split
X_train,X_test,y_train,y_test=train_test_split(df['reviewText'],df['rating'],test_size=0.2,random_state=42)

In [47]:
from sklearn.feature_extraction.text import CountVectorizer
bow =CountVectorizer()
X_train_bow= bow.fit_transform(X_train).toarray()
X_test_bow= bow.transform(X_test).toarray()

In [48]:
from sklearn.feature_extraction.text import TfidfVectorizer
tf_idf =TfidfVectorizer()
X_train_tf_idf= tf_idf.fit_transform(X_train).toarray()
X_test_tf_idf= tf_idf.transform(X_test).toarray()

In [45]:
X_train_bow

<Compressed Sparse Row sparse matrix of dtype 'int64'
	with 432286 stored elements and shape (9600, 35736)>

In [49]:
from sklearn.naive_bayes import GaussianNB
from sklearn.metrics import accuracy_score,confusion_matrix,classification_report
nb_model_bow=GaussianNB().fit(X_train_bow,y_train)
nb_model_tfidf=GaussianNB().fit(X_train_tf_idf,y_train)

In [50]:

y_pred_bow=nb_model_bow.predict(X_test_bow)

In [52]:
y_pred_tfidf=nb_model_bow.predict(X_test_tf_idf)

In [53]:
confusion_matrix(y_test,y_pred_bow)

array([[499, 304],
       [717, 880]])

In [54]:
print("BOW accuracy: ",accuracy_score(y_test,y_pred_bow))


BOW accuracy:  0.5745833333333333


In [55]:
confusion_matrix(y_test,y_pred_tfidf)

array([[490, 313],
       [710, 887]])

In [56]:
print("TFIDF accuracy: ",accuracy_score(y_test,y_pred_tfidf))

TFIDF accuracy:  0.57375


In [58]:
!pip install gensim
import gensim
from gensim.models import Word2Vec, KeyedVectors

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 27.8/27.8 MB 23.4 MB/s eta 0:00:00


In [66]:
sentences = X_train.fillna("").astype(str).apply(str.split).tolist()

w2v_model = Word2Vec(
    sentences=sentences,
    vector_size=100,
    window=5,
    min_count=2,
    workers=4,
    epochs=20
)

In [67]:
import numpy as np

def get_vector(review):
    words = str(review).split()

    vectors = [w2v_model.wv[word] for word in words if word in w2v_model.wv]

    if not vectors:
        return np.zeros(100)

    return np.mean(vectors, axis=0)

In [68]:
X_train_w2v = np.array([
    get_vector(review) for review in X_train.fillna("")
])

X_test_w2v = np.array([
    get_vector(review) for review in X_test.fillna("")
])

In [70]:
from sklearn.naive_bayes import GaussianNB
from sklearn.metrics import (
    accuracy_score,
    confusion_matrix,
    classification_report
)

nb_model_w2v = GaussianNB().fit(
    X_train_w2v,
    y_train
)

In [71]:
y_pred_w2v = nb_model_w2v.predict(X_test_w2v)


In [73]:
confusion_matrix(y_test,y_pred_w2v)

array([[ 625,  178],
       [ 395, 1202]])

In [72]:
print("Word2Vec Accuracy:",
      accuracy_score(y_test, y_pred_w2v))


Word2Vec Accuracy: 0.76125


In [74]:
from sklearn.metrics import classification_report

print(classification_report(y_test, y_pred_w2v))

              precision    recall  f1-score   support

           0       0.61      0.78      0.69       803
           1       0.87      0.75      0.81      1597

    accuracy                           0.76      2400
   macro avg       0.74      0.77      0.75      2400
weighted avg       0.78      0.76      0.77      2400

